# Univariate LSTM Model

Reference: Jason Brownlee. "Deep Learning for Time Series Forecasting: Predict the Future with MLPs, CNNs, and LSTMs in Python".

>LSTMs can be used to model univariate time series forecasting problems. These are problems
comprised of a single series of observations and a model is required to learn from the series of
past observations to predict the next value in the sequence. We will demonstrate a number of
variations of the LSTM model for univariate time series forecasting. This section is divided into
six parts; they are:
>1. Data Preparation
>2. Vanilla LSTM
>3. Stacked LSTM
>4. Bidirectional LSTM
>5. CNN-LSTM
>6. ConvLSTM
Each of these models are demonstrated for one-step univariate time series forecasting, but
can easily be adapted and used as the input part of a model for other types of time series
forecasting problems.

In [1]:
# univariate data preparation
from numpy import array

# split a univariate sequence into samples
def split_sequence(sequence, n_steps):
    X, y = list(), list()
    for i in range(len(sequence)):
        # find the end of this pattern
        end_ix = i + n_steps
        # check if we are beyond the sequence
        if end_ix > len(sequence)-1:
            break
        # gather input and output parts of the pattern
        seq_x, seq_y = sequence[i:end_ix], sequence[end_ix]
        X.append(seq_x)
        y.append(seq_y)
    return array(X), array(y)

# define input sequence
raw_seq = [10, 20, 30, 40, 50, 60, 70, 80, 90]
# choose a number of time steps
n_steps = 3
# split into samples
X, y = split_sequence(raw_seq, n_steps)
# summarize the data
for i in range(len(X)):
    print(X[i], y[i])

[10 20 30] 40
[20 30 40] 50
[30 40 50] 60
[40 50 60] 70
[50 60 70] 80
[60 70 80] 90


>A Vanilla LSTM is an LSTM model that has a single hidden layer of LSTM units, and an
output layer used to make a prediction. Key to LSTMs is that they oﬀer native support for
sequences. Unlike a CNN that reads across the entire input vector, the LSTM model reads one
time step of the sequence at a time and builds up an internal state representation that can be
used as a learned context for making a prediction. We can define a Vanilla LSTM for univariate
time series forecasting as follows.

In [2]:
# reshape from [samples, timesteps] into [samples, timesteps, features]
n_features = 1
X = X.reshape((X.shape[0], X.shape[1], n_features))

In [3]:
# Modern Keras imports
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input

model = Sequential()
model.add(Input(shape=(n_steps, n_features)))
model.add(LSTM(50, activation='relu', return_sequences=True))
model.add(LSTM(50, activation='relu'))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')

/Users/thomschu/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [4]:
# fit model
model.fit(X, y, epochs=200, verbose=0)

In [6]:
# demonstrate prediction
x_input = array([70, 80, 90])
x_input = x_input.reshape((1, n_steps, n_features))
yhat = model.predict(x_input, verbose=0)
print(f"Predicted value: {yhat}")

Predicted value: [[103.68442]]


In [7]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 3, 50)          │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 91,955 (359.20 KB)

 Trainable params: 30,651 (119.73 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 61,304 (239.47 KB)


### 1. The "Output Shape" Column
This column tracks the shape of the data as it exits each layer. The word **`None`** just stands for the number of **Samples** (it is `None` because the model can accept any number of samples you give it).

* **`lstm (LSTM)` -> `(None, 3, 50)`:** Here is the proof of `return_sequences=True` in action! Because we set that to True, the first LSTM keeps the **3 time steps** intact. It looked at the data and extracted **50** complex features for *each* of those 3 steps.
* **`lstm_1 (LSTM)` -> `(None, 50)`:** Notice how the `3` is suddenly gone? Because we did *not* use `return_sequences=True` here, this second LSTM read all 3 time steps and condensed them down into a single, final summary of **50** numbers.
* **`dense (Dense)` -> `(None, 1)`:** The Dense layer takes those 50 summary numbers and crushes them down into your **1** final predicted value.

### 2. The "Param #" Column
"Param" stands for **Parameters**. You can explain parameters to students as the **"knobs and dials"** the neural network is allowed to twist and tune while it is learning. 

* The **Dense** layer only has 51 parameters (it connects its 1 neuron to the 50 outputs from the previous layer, plus 1 bias term).
* The **LSTM** layers have *tens of thousands* of parameters (10,400 and 20,200). Why? Because LSTMs are complex! They contain internal "memory gates" (forget gates, input gates, output gates) that have to learn what information to keep, what to throw away, and what to pass on over time.

### 3. Total Params
At the bottom, it shows **Total params: 91,955**. This means that when you eventually run `model.fit()`, the Adam optimizer is going to be fine-tuning over 90,000 tiny mathematical dials to try and make your Mean Squared Error (MSE) as close to zero as possible!